# Load Dataset

In [ ]:
import os
import json
import random
import numpy as np

from dotenv import load_dotenv 

#Initialize constants
DATASET_PATH = "../ConvQuestions/test_set/test_set_ALL.json"
random_state = 1
random.seed(random_state)
np.random.seed(random_state)
NUM_SAMPLES = 100
REPETITIONS = 3
load_dotenv("../.env") #Load API key from .env file
API_KEY = os.environ["GROQ_API_KEY"]
MODEL = os.environ["MODEL"]
MAX_TOKENS = int(os.environ["TOKENS_PER_REQUEST"])
DELAY = float(os.environ["REQUEST_DELAY"])

#Caching so that we can either view stuff at intermediate stages or save on redoing steps
QUERY_CACHE_PATH = "cached_queries.json" #For saving on API query costs so we don't rerun them every time.
USE_QUERY_CACHE = False
METRICS_CACHE_PATH = "cached_metrics.json" #Show individual metrics for each question
USE_METRICS_CACHE = False


#Load dataset and then sample conversations
with open(DATASET_PATH, 'r') as f:
    dataset = json.load(f)
sampled_convos = random.sample(dataset, NUM_SAMPLES)

# Model Queries

In [ ]:
from groq import Groq
import time

groq_client = Groq(api_key=API_KEY)

def query_model(convo, REPETITIONS):
    #Initialize empty list for answers and latencies in each question
    for question in convo["questions"]:
        question["given_answers"] = []
        question["latencies"] = []
    
    #Run through each conversation multiple times so we can check consistency later
    for i in range(REPETITIONS):
        history = []
        #Query the model and update history on each question
        for question in convo["questions"]:
            time.sleep(DELAY) #Rate limiting
            question_text = question["question"]
            history.append({"role": "user", "content": question_text})

            start_time = time.time()
            response = groq_client.chat.completions.create(model=MODEL, messages=history, max_tokens=MAX_TOKENS)
            end_time = time.time()

            extracted_response = response.choices[0].message.content
            history.append({"role":"assistant", "content": extracted_response})
            question["given_answers"].append(extracted_response)
            question["latencies"].append(end_time - start_time)

#We modify the convo data in place to insert the query data and "answer" them with our LLM
answered_queries = sampled_convos
if USE_QUERY_CACHE and os.path.isfile(QUERY_CACHE_PATH):
    with open(QUERY_CACHE_PATH, "r") as file:
        answered_queries = json.load(file)
else:
    for query in answered_queries:
        query_model(query, REPETITIONS)
    with open(QUERY_CACHE_PATH, "w") as file:
        json.dump(answered_queries, file)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kpkx22nhfv48ag85dmdcp7hw` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199998, Requested 178. Please try again in 1m16.032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

# Evaluate Metrics Functions (P@1 Accuracies, Consistency Score, Avg. Response Latency)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import re
import spacy

#Simply average together all the query latencies
def eval_avg_latency():
    latencies = []
    questions = [question for convo in answered_queries for question in convo["questions"]]
    for question in questions:
        avg_latency = sum(question["latencies"]) / REPETITIONS
        question["avg_latency"] = avg_latency
        latencies.append(avg_latency)
    avg_latency = sum(latencies) / len(latencies)
    return avg_latency

sentence_transformer_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
def similarity_helper(sentences):
    embeddings = sentence_transformer_model.encode(sentences)
    centroid = np.mean(embeddings, axis=0)
    similarities = cosine_similarity(embeddings, [centroid])
    return float(np.mean(similarities))
def eval_consistency():
    consistency_scores = []
    questions = [question for convo in answered_queries for question in convo["questions"]]
    for question in questions:
        consistency_score = similarity_helper(question["given_answers"])
        question["consistency_score"] = consistency_score
        consistency_scores.append(consistency_score)
    return sum(consistency_scores) / len(consistency_scores)

spacy_model = spacy.load("en_core_web_sm")
similarity_threshold = 0.8
def accuracy_helper(expected, given, full_question):
    #Start off by normalizing the texts
    expected = re.sub(r"[^\w\s]", "", expected.lower())
    given = re.sub(r"[^\w\s]", "", given.lower())

    #Make sure that the set of entities in expected is a subset of given
    expected_entities = {entity.text.lower() for entity in spacy_model(expected).ents}
    given_entities = {entity.text.lower() for entity in spacy_model(given).ents}
    match = expected_entities.issubset(given_entities)
    if match:
        return 1
    
    #Fallback to semantic similarity in case we don't get a match. Include the full question so the model actually knows what the context for the expected answer is.
    similarity_score = similarity_helper([f"{full_question} {expected}", given])
    match = similarity_score >= similarity_threshold
    if match:
        return 1
    
    return 0
def eval_accuracies():
    accuracies = []
    accuracies_multihop = []
    questions = [question for convo in answered_queries for question in convo["questions"]]
    for question in questions:
        question["accuracies"] = []
        for given_answer in question["given_answers"]:
            full_question = question["completed_question"] if "completed_question" in question else question["question"]
            accuracy = accuracy_helper(question["answer_text"], given_answer, full_question)
            question["accuracies"].append(accuracy)
        question["avg_accuracy"] = sum(question["accuracies"]) / len(question["accuracies"])
        accuracies.append(question["avg_accuracy"])
        if question["turn"] != 0:
            accuracies_multihop.append(question["avg_accuracy"])
    
    accuracy = sum(accuracies) / len(accuracies)
    accuracy_multihop = sum(accuracies_multihop) / len(accuracies_multihop)
    return accuracy, accuracy_multihop



# Compile Metrics

In [ ]:
#Note we put together queries with intermediate computations and metrics obj together in same file with metrics at the top for easier viewing.
metrics_obj = None
if USE_METRICS_CACHE and os.path.isfile(METRICS_CACHE_PATH):
    with open(METRICS_CACHE_PATH, "r") as file:
        metrics_obj = json.load(file)[0]
else:
    p1_accuracy, p1_accuracy_multihop = eval_accuracies()
    consistency_score = eval_consistency()
    avg_response_latency = eval_avg_latency()
    metrics_obj = {"Overall P@1": p1_accuracy, "Multi-hop P@1": p1_accuracy_multihop, "Consistency Score": consistency_score, "Avg. Response Latency": avg_response_latency}
    answered_queries.insert(0, metrics_obj)
    with open(METRICS_CACHE_PATH, "w") as file:
        json.dump(answered_queries, file)

#Now print out all of our compiled Metrics
for key, value in metrics_obj.items():
    print(f"{key}: {value}")